# C4 – learn it step by step

Run each cell with **Shift + Enter**, top to bottom. This notebook only *explores*; the real system lives in `src/`.
Before starting, run `python -m src.bootstrap.train_and_register` once from the project root (see README).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.common.config import LABEL, all_features, feature_lists
from src.bootstrap.data import load_split
print("Project root:", ROOT)

## 1. Look at the data
Each row is one person. The **features** are the inputs; the **label** `y` is 1 when income is above 50K.

In [ ]:
train = load_split("train")
num, cat = feature_lists()
print(train.shape)
print("Numeric:", num)
print("Categorical:", cat)
print("Share earning >50K:", round(train[LABEL].mean(), 3))
train.head()

In [ ]:
train[num].describe().round(1)

In [ ]:
train[cat].isna().mean().round(3)   # missing values per categorical column

## 2. Train a model yourself (what 'training' means)
1. Build the preprocessing pipeline (fill gaps, scale numbers, one-hot encode categories).
2. `fit` = learn from the training rows.
3. `predict` on rows the model has never seen, and measure how often it is right.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from src.bootstrap.train_and_register import make_preprocessor

test = load_split("test")
pipe = Pipeline([("preprocess", make_preprocessor()), ("model", LogisticRegression(max_iter=1000))])
pipe.fit(train[all_features()], train[LABEL])            # <- this line IS the training

proba = pipe.predict_proba(test[all_features()])[:, 1]   # confidence that y = 1
pred = (proba >= 0.5).astype(int)
print("accuracy", round(accuracy_score(test[LABEL], pred), 3))
print("F1      ", round(f1_score(test[LABEL], pred), 3))
print("ROC-AUC ", round(roc_auc_score(test[LABEL], proba), 3))

In [ ]:
plt.hist(proba, bins=20)
plt.title("Model confidence on the test set (C4 monitors this)")
plt.xlabel("predicted probability of >50K"); plt.show()

## 3. The versioned baseline that C4 compares against

In [ ]:
from src.common.config import load_json, model_dir
from src.common import registry
v = registry.get_alias_version("production")
baseline = load_json(model_dir(v) / "baseline.json")
print("Production model version:", v, baseline["model_type"])
print("Baseline performance:", {k: round(x, 3) for k, x in baseline["performance"].items() if k != "n"})
pd.Series(baseline["feature_importance"]).sort_values().plot.barh(title="Feature importance")
plt.show()

## 4. Drift tests by hand: PSI and KS on one feature
Compare a clean window with a drifted window from scenario S2 (age shifted up).

In [ ]:
from scipy.stats import ks_2samp
from src.c4.monitoring.stats import bin_shares, psi
from src.common.config import DATA_SCENARIOS

s2 = pd.read_csv(DATA_SCENARIOS / "S2_covariate_degrading" / "windows.csv")
clean, drifted = s2[s2.window == 5], s2[s2.window == 20]
b = baseline["numeric"]["age"]
for name, w in [("clean window 5", clean), ("drifted window 20", drifted)]:
    p = psi(b["bin_shares"], bin_shares(w["age"], b["bin_edges"]))
    ks = ks_2samp(load_split("val")["age"], w["age"])
    print(f"{name:<18} PSI={p:.3f}  KS stat={ks.statistic:.3f}  p-value={ks.pvalue:.4f}")

In [ ]:
plt.hist(clean["age"], bins=30, alpha=0.6, label="window 5 (clean)")
plt.hist(drifted["age"], bins=30, alpha=0.6, label="window 20 (drifted)")
plt.legend(); plt.title("age distribution"); plt.show()

## 5. Calibrated thresholds
Each feature has its own threshold learned from clean data (mean + 3 × std).

In [ ]:
thr = load_json(model_dir(v) / "thresholds.json")
pd.Series(thr["features"]).round(3)

## 6. Look at a C4 run
Run `python -m src.c4.run_c4 --scenario S3_concept` first, then:

In [ ]:
from src.common.config import OUTPUTS
log = pd.read_csv(OUTPUTS / "runs" / "S3_concept" / "window_log.csv")
log[["window", "n_fired", "f1", "perf_drop", "diagnosis", "severity_level", "severity_score", "action"]].iloc[12:32]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(log.window, log.f1, marker="o", label="F1")
ax.axvline(15, color="red", ls="--", label="drift injected")
ax.axhline(baseline["performance"]["f1"], color="gray", ls=":", label="baseline F1")
ax.legend(); ax.set_xlabel("window"); plt.show()